Code for **"Inpainting"** figures $6$, $8$ and 7 (top) from the main paper.

In [ ]:
"""
*Uncomment if running on colab*
Set Runtime -> Change runtime type -> Under Hardware Accelerator select GPU in Google Colab
"""
!git clone https://github.com/DmitryUlyanov/deep-image-prior
!mv deep-image-prior/* ./

# Import libs

In [ ]:
from __future__ import print_function
import matplotlib.pyplot as plt
%matplotlib inline

import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import numpy as np
from models.resnet import ResNet
from models.unet import UNet
from models.skip import skip
import torch
import torch.optim

from utils.inpainting_utils import *

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark =True
dtype = torch.cuda.FloatTensor

PLOT = True
imsize = -1
dim_div_by = 64

# Choose figure

In [ ]:
img_path  = 'data/inpainting/naip.tif'
mask_path = 'data/inpainting/mask.png'

NET_TYPE = 'skip_depth6' # one of skip_depth4|skip_depth2|UNET|ResNet

# Load mask

In [ ]:
import rasterio
import numpy as np

with rasterio.open(img_path) as src:
    img_np = src.read().astype(np.float32)

# normalize 每個 band 到 0~1
for c in range(img_np.shape[0]):
    band = img_np[c]
    low = np.percentile(band, 2)
    high = np.percentile(band, 98)
    img_np[c] = np.clip(
        (band - low) / (high - low + 1e-8),
        0,
        1
    )

print(img_np.shape)
# 從大圖中央裁 448*448
crop_size = 448

H = img_np.shape[1]
W = img_np.shape[2]

y0 = (H - crop_size) // 2
x0 = (W - crop_size) // 2

img_np = img_np[
    :,
    y0:y0+crop_size,
    x0:x0+crop_size
]

print(img_np.shape)
img_mask_pil, img_mask_np = get_image(mask_path, imsize)

### Center crop

In [ ]:
img_mask_pil = crop_image(img_mask_pil, dim_div_by)

img_mask_np = pil_to_np(img_mask_pil)

### Visualize

In [ ]:
img_mask_var = np_to_torch(img_mask_np).type(dtype)

import matplotlib.pyplot as plt
import numpy as np

# RGB
rgb = img_np[:3]

# NIR
nir = img_np[3]

# mask
if img_mask_np.shape[0] > 1:
    mask_show = img_mask_np[0]
else:
    mask_show = img_mask_np.squeeze()

# masked RGB
masked_rgb = rgb * img_mask_np[:1]

# masked NIR
masked_nir = nir * mask_show

plt.figure(figsize=(16,8))

plt.subplot(2,3,1)
plt.imshow(np.clip(rgb.transpose(1,2,0), 0, 1))
plt.title("RGB")
plt.axis("off")

plt.subplot(2,3,2)
plt.imshow(mask_show, cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.subplot(2,3,3)
plt.imshow(np.clip(masked_rgb.transpose(1,2,0), 0, 1))
plt.title("Masked RGB")
plt.axis("off")

plt.subplot(2,3,4)
plt.imshow(nir, cmap="gray")
plt.title("NIR")
plt.axis("off")

plt.subplot(2,3,5)
plt.imshow(mask_show, cmap="gray")
plt.title("Mask")
plt.axis("off")

plt.subplot(2,3,6)
plt.imshow(masked_nir, cmap="gray")
plt.title("Masked NIR")
plt.axis("off")

plt.tight_layout()
plt.show()

# Setup

In [ ]:
pad = 'reflection' # 'zero'
OPT_OVER = 'net'
OPTIMIZER = 'adam'

In [ ]:
if 'naip.tif' in img_path:
  # Same params and net as in super-resolution and denoising
  INPUT = 'noise'
  input_depth = 32
  LR = 0.01
  num_iter = 6001
  param_noise = False
  show_every = 50
  figsize = 5
  reg_noise_std = 0.00

  net = skip(input_depth, img_np.shape[0],
              num_channels_down = [16, 32, 64, 128, 128],
              num_channels_up =   [16, 32, 64, 128, 128],
              num_channels_skip =    [0] * 5,
              filter_size_up = 3, filter_size_down = 3,
              upsample_mode='nearest', filter_skip_size=1,
              need_sigmoid=True, need_bias=True, pad=pad, act_fun='LeakyReLU').type(dtype)


else:
    assert False

net = net.type(dtype)
net_input = get_noise(input_depth, INPUT, img_np.shape[1:]).type(dtype)

In [ ]:
# Compute number of parameters
s  = sum(np.prod(list(p.size())) for p in net.parameters())
print ('Number of params: %d' % s)

# Loss
mse = torch.nn.MSELoss().type(dtype)

img_var = np_to_torch(img_np).type(dtype)
mask_var = np_to_torch(img_mask_np).type(dtype)

# Main loop

In [ ]:
i = 0
def closure():

    global i

    if param_noise:
        for n in [x for x in net.parameters() if len(x.size()) == 4]:
            n = n + n.detach().clone().normal_() * n.std() / 50

    net_input = net_input_saved
    if reg_noise_std > 0:
        net_input = net_input_saved + (noise.normal_() * reg_noise_std)


    out = net(net_input)

    total_loss = mse(out * mask_var, img_var * mask_var)
    total_loss.backward()

    print ('Iteration %05d    Loss %f' % (i, total_loss.item()), '\r', end='')
    if PLOT and i % show_every == 0:
      out_np = torch_to_np(out)
      out_np = np.clip(out_np, 0, 1)

      # 前三個 channels = RGB
      rgb_out = out_np[:3]

      # 第四個 channel = NIR
      nir_out = out_np[3]

      import matplotlib.pyplot as plt

      plt.figure(figsize=(12, 5))

      # RGB
      plt.subplot(1, 2, 1)
      plt.imshow(rgb_out.transpose(1, 2, 0))

      # NIR
      plt.subplot(1, 2, 2)
      plt.imshow(nir_out, cmap='gray')

      plt.tight_layout()
      plt.show()
    i += 1

    return total_loss

net_input_saved = net_input.detach().clone()
noise = net_input.detach().clone()

p = get_params(OPT_OVER, net, net_input)
optimize(OPTIMIZER, p, closure, LR, num_iter)

In [ ]:
out_np = torch_to_np(net(net_input))
out_np = np.clip(out_np, 0, 1)

# 前三個 channel = RGB
rgb_out = out_np[:3]

# 第四個 channel = NIR
nir_out = out_np[3]

plt.figure(figsize=(12, 5))

# RGB
plt.subplot(1, 2, 1)
plt.imshow(rgb_out.transpose(1, 2, 0))
plt.title('DIP Restored RGB')
plt.axis('off')

# NIR
plt.subplot(1, 2, 2)
plt.imshow(nir_out, cmap='gray')
plt.title('DIP Restored NIR')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')